# London Transport Network 

Models the London contactless network as a graph to find the most likely interchange points between DLR and other services based on shortest journey time. 

### Key Concepts:
* **NLC (National Location Code):** Represents a set of physical ticket gates. Separates multi-modal stations (e.g., London Bridge NR vs. Underground) into distinct graph nodes.

* **ICS:** TfL’s station-level identifier used for Journey Planner API lookups.

* **NaPTAN (National Public Transport Access Node)**: A national standard system identifier used across the UK transport network to provide precise station-level granularity. The code is typically prefixed with 940GZZLU for Tube, 940GZZDL for DLR, or 910G/490G for National Rail stations.

**Output:** A comprehensive mapping table of all DLR-to-non-DLR journeys with optimal in-station interchange points.

In [ ]:
# ==============================================================================
# LIBRARY IMPORTS & ENVIRONMENT SETUP
# ==============================================================================

# File management, environment variables, and graph serialization
import os
import pickle
from dotenv import load_dotenv

# Tabular data manipulation and numerical operations
import numpy as np
import pandas as pd

# Python built-ins for data processing and rate limiting
from itertools import combinations
import json
import re
import time

# Querying TfL API endpoints
import requests
import urllib

# Constructing network topology and running Dijkstra shortest path algotrithm
import networkx as nx

# Load API key from environment (.env file)
load_dotenv()
TFL_API_KEY = os.getenv("TFL_API_KEY", "")

if not TFL_API_KEY:
  print(
      "Warning: TFL_API_KEY not found in environment variables. API calls will"
      " rely on unauthenticated rate limits."
  )
else:
  print("TFL_API_KEY successfully loaded.")

TFL_API_KEY successfully loaded.


## Network Nodes - All Contactless Stations

Loads all ~860 contactless train stations across National Rail, Tube, DLR, Overground, and Elizabeth line. <br>
_Bus, tram, and river services are excluded._

Each station row maps its unique **NLC** (gate line) to its corresponding **ICS** station identifier.

In [ ]:
all_stations = pd.read_csv("all_stations.csv") #load station nodes
print(
    f"Loaded {len(all_stations)} stations across {all_stations['NLC'].nunique()}"
    " unique NLC gate lines."
)
all_stations.head()

Loaded 860 stations across 860 unique NLC gate lines.


,NLC,STATION,INNER_ZONE,OUTER_ZONE,JOINT,GATED
0,750,Abbey Road DLR,2,3,TfL,Ungated
1,5131,Abbey Wood,4,4,NR,Gated
2,1404,Acton Central,3,3,NR,Gated
3,3000,Acton Main Line,3,3,NR,Gated
4,500,Acton Town,3,3,TfL,Gated


In [23]:
# ==============================================================================
# RATE-LIMITED MASTER REFERENCE BUILDER (ICS + NaPTAN + EXPONENTIAL BACKOFF)
# ==============================================================================

INPUT_FILE = "all_stations.csv"
OUTPUT_FILE = "all_stations_ref.csv"
TFL_APP_KEY = (
    TFL_API_KEY if "TFL_API_KEY" in globals() else None
)  # Uses environment key if available

# Manual ICS map for complex hubs/platform splits[cite: 2]
MANUAL_ICS_MAP = {
    5667: "1000009",
    1090: "1000010",
    3176: "1000036",
    6560: "1000038",
    852: "1000038",
    6561: "1000058",
    5357: "1000067",
    6000: "1000068",
    5414: "1000070",
    774: "1000072",
    569: "1000072",
    773: "1000099",
    593: "1000099",
    5050: "1000105",
    7090: "1000106",
    7091: "1000107",
    9846: "1000108",
    618: "1000125",
    600: "1000129",
    625: "1000129",
    772: "1000129",
    3171: "1000133",
    689: "1000139",
    3667: "1000140",
    9227: "1000141",
    6971: "1000159",
    5376: "1000160",
    6552: "1000174",
    8204: "1000174",
    6093: "1000174",
    5424: "1000185",
    5596: "1000186",
    7454: "1000187",
    5431: "1000189",
    5380: "1000198",
    3061: "1000201",
    1082: "1000203",
    9587: "1000205",
    5074: "1000207",
    6821: "1000216",
    5604: "1000217",
    7422: "1000218",
    7423: "1000218",
    7222: "1000222",
    5671: "1000224",
    5385: "1000226",
    5437: "1000228",
    559: "1000262",
    747: "1000254",
    796: "1000254",
    5598: "1000254",
    5465: "1000254",
    748: "1000256",
    1421: "1000258",
    6562: "1000267",
}


def get_identifiers_with_backoff(
    station_name: str, app_key: str = None, retries: int = 4
) -> tuple:
  """Queries TfL StopPoint Search API with exponential backoff on 429 rate limits or timeouts."""
  clean_name = (
      station_name.replace(" DLR", "")
      .replace(" NR", "")
      .replace(" LU", "")
      .replace(" LO", "")
      .replace(" Elizabeth Line", "")
      .replace(" Station", "")
      .strip()
  )

  encoded = urllib.parse.quote(clean_name)
  params = {"modes": "national-rail,tube,dlr,overground,elizabeth-line"}
  if app_key:
    params["app_key"] = app_key

  url = f"https://api.tfl.gov.uk/StopPoint/Search/{encoded}?{urllib.parse.urlencode(params)}"
  req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})

  delay = 2.0  # Initial retry delay

  for attempt in range(retries):
    try:
      with urllib.request.urlopen(req, timeout=8) as resp:
        data = json.loads(resp.read().decode())
        matches = data.get("matches", [])

        found_ics = None
        found_naptan = None

        for match in matches:
          if not found_ics:
            ics = match.get("icsId") or match.get("topMostParentId")
            if ics and str(ics).isdigit():
              found_ics = str(ics)

          if not found_naptan:
            match_id = match.get("id", "")
            if (
                match_id.startswith("940")
                or match_id.startswith("910")
                or match_id.startswith("490")
            ):
              found_naptan = match_id

          if found_ics and found_naptan:
            break

        return found_ics, found_naptan

    except urllib.error.HTTPError as e:
      if e.code == 429:
        time.sleep(delay)
        delay *= 2  # Double delay on rate limit
      else:
        break
    except Exception:
      time.sleep(delay)
      delay *= 2  # Backoff on connection timeouts

  return None, None


# Execution Loop
df = pd.read_csv(INPUT_FILE)
print(f"Starting rate-limited processing for {len(df)} stations...")

if "ICS" not in df.columns:
  df["ICS"] = ""
if "NaPTAN" not in df.columns:
  df["NaPTAN"] = ""

ics_list = []
naptan_list = []

for idx, row in df.iterrows():
  station_name = str(row["STATION"])
  nlc_int = int(row["NLC"])

  # Check manual map first[cite: 2]
  ics = MANUAL_ICS_MAP.get(nlc_int, str(row.get("ICS", "")) if pd.notna(row.get("ICS")) else "")
  naptan = str(row.get("NaPTAN", "")) if pd.notna(row.get("NaPTAN")) else ""

  # If either code is missing, query API with rate limiting
  if not ics or not naptan or ics == "nan" or naptan == "nan":
    api_ics, api_naptan = get_identifiers_with_backoff(
        station_name, app_key=TFL_APP_KEY
    )
    if not ics or ics == "nan":
      ics = api_ics
    if not naptan or naptan == "nan":
      naptan = api_naptan

  ics_list.append(ics if ics else "")
  naptan_list.append(naptan if naptan else "")

  # Explicit rate limit pause (~100-120 requests per minute to stay safe)
  time.sleep(0.5)

  if (idx + 1) % 50 == 0:
    print(
        f"  Progress: {idx + 1}/{len(df)} stations processed."
    )

df["ICS"] = ics_list
df["NaPTAN"] = naptan_list

# Clean Name Column generation
df["CLEAN STATION NAME"] = (
    df["STATION"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.replace(" DLR", "", regex=False)
    .str.replace(" LU", "", regex=False)
    .str.replace(" NR", "", regex=False)
    .str.replace(" LO", "", regex=False)
    .str.replace(" Station", "", regex=False)
    .str.strip()
    .str.title()
)

cols = [
    "ICS",
    "NaPTAN",
    "NLC",
    "STATION",
    "CLEAN STATION NAME",
    "INNER_ZONE",
    "OUTER_ZONE",
    "JOINT",
    "GATED",
]
df_master = df[[c for c in cols if c in df.columns]]
df_master.to_csv(OUTPUT_FILE, index=False)

print(f"\nProcessing Complete! Saved to '{OUTPUT_FILE}'.")
print(f"Total Stations: {len(df_master)}")
print(f"ICS Populated: {(df_master['ICS'] != '').sum()}")
print(f"NaPTAN Populated: {(df_master['NaPTAN'] != '').sum()}")
df_master.head(10)

Starting rate-limited processing for 860 stations...


  Progress: 50/860 stations processed (Paused safely).
  Progress: 100/860 stations processed (Paused safely).
  Progress: 150/860 stations processed (Paused safely).
  Progress: 200/860 stations processed (Paused safely).
  Progress: 250/860 stations processed (Paused safely).
  Progress: 300/860 stations processed (Paused safely).
  Progress: 350/860 stations processed (Paused safely).
  Progress: 400/860 stations processed (Paused safely).
  Progress: 450/860 stations processed (Paused safely).
  Progress: 500/860 stations processed (Paused safely).
  Progress: 550/860 stations processed (Paused safely).
  Progress: 600/860 stations processed (Paused safely).
  Progress: 650/860 stations processed (Paused safely).
  Progress: 700/860 stations processed (Paused safely).
  Progress: 750/860 stations processed (Paused safely).
  Progress: 800/860 stations processed (Paused safely).
  Progress: 850/860 stations processed (Paused safely).

Processing Complete! Saved to 'all_stations_mast

,ICS,NaPTAN,NLC,STATION,CLEAN STATION NAME,INNER_ZONE,OUTER_ZONE,JOINT,GATED
0,1003006,940GZZDLABR,750,Abbey Road DLR,Abbey Road,2,3,TfL,Ungated
1,1001001,910GFILTNEW,5131,Abbey Wood,Abbey Wood,4,4,NR,Gated
2,1001002,910GACTNCTL,1404,Acton Central,Acton Central,3,3,NR,Gated
3,1001003,910GACTONML,3000,Acton Main Line,Acton Main Line,3,3,NR,Gated
4,1000002,940GZZLUACT,500,Acton Town,Acton Town,3,3,TfL,Gated
5,1001275,910GADLESTN,5550,Addlestone,Addlestone,H,H,NR,Gated
6,1001004,910GALBNYPK,5132,Albany Park,Albany Park,5,5,NR,Gated
7,1000004,940GZZLUADE,502,Aldgate,Aldgate,1,1,TfL,Gated
8,1000004,940GZZLUADE,503,Aldgate East,Aldgate East,1,1,TfL,Gated
9,1001005,910GALEXNDP,6025,Alexandra Palace,Alexandra Palace,3,3,NR,Gated


In [ ]:
# ==============================================================================
# RETRY FAILED LOOKUPS & VALIDATE JOINT TYPES AGAINST NaPTAN PREFIXES
# ==============================================================================

# Load existing master reference table
df_ref = pd.read_csv("all_stations_ref.csv")
df_ref["ICS"] = df_ref["ICS"].fillna("").astype(str)
df_ref["NaPTAN"] = df_ref["NaPTAN"].fillna("").astype(str)

print(
    f"Before Retry -> ICS Populated: {(df_ref['ICS'] != '').sum()} | NaPTAN"
    f" Populated: {(df_ref['NaPTAN'] != '').sum()}"
)

# RETRY LOOP FOR MISSING CODES
for idx, row in df_ref.iterrows():
  ics = row["ICS"]
  naptan = row["NaPTAN"]

  if (ics and ics != "nan") and (naptan and naptan != "nan"):
    continue

  station_name = str(row["STATION"])
  # Relaxed query cleaning (stripping brackets and operator tags)
  clean_query = (
      station_name.split("(")[0]
      .replace(" DLR", "")
      .replace(" LU", "")
      .replace(" NR", "")
      .replace(" LO", "")
      .strip()
  )

  url = f"https://api.tfl.gov.uk/StopPoint/Search/{urllib.parse.quote(clean_query)}?modes=national-rail,tube,dlr,overground,elizabeth-line"

  try:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=5) as resp:
      data = json.loads(resp.read().decode())
      matches = data.get("matches", [])
      for match in matches:
        if not ics or ics == "nan":
          found_ics = match.get("icsId") or match.get("topMostParentId")
          if found_ics and str(found_ics).isdigit():
            ics = str(found_ics)
        if not naptan or naptan == "nan":
          match_id = match.get("id", "")
          if (
              match_id.startswith("940")
              or match_id.startswith("910")
              or match_id.startswith("490")
          ):
            naptan = match_id
        if ics and naptan:
          break
  except Exception:
    pass

  df_ref.at[idx, "ICS"] = ics if ics and ics != "nan" else ""
  df_ref.at[idx, "NaPTAN"] = naptan if naptan and naptan != "nan" else ""
  time.sleep(0.04)  # Rate limiting pause

print(
    f"After Retry -> ICS Populated: {(df_ref['ICS'] != '').sum()} | NaPTAN"
    f" Populated: {(df_ref['NaPTAN'] != '').sum()}"
)


# VALIDATION FUNCTION: Check JOINT column vs NaPTAN Prefix
def validate_station_type(row):
  joint = str(row["JOINT"]).strip().lower()
  naptan = str(row["NaPTAN"]).strip()

  if not naptan:
    return "Missing NaPTAN"

  is_tfl = naptan.startswith("940")
  is_nr = naptan.startswith("910") or naptan.startswith("490")

  if joint == "tfl" and is_tfl:
    return "Match"
  elif joint == "nr" and is_nr:
    return "Match"
  elif joint == "joint":
    return "Joint Hub"
  else:
    return "Review / Mismatch"


df_ref["Validation_Check"] = df_ref.apply(validate_station_type, axis=1)

# Save updated master reference table
df_ref.to_csv("all_stations_ref.csv", index=False)

print("\nValidation Summary (JOINT vs. NaPTAN Prefix):")
print(df_ref["Validation_Check"].value_counts())
df_ref[df_ref["Validation_Check"] == "Review / Mismatch"][
    ["STATION", "JOINT", "NaPTAN"]
].head(10)

Before Retry -> ICS Populated: 860 | NaPTAN Populated: 712
After Retry -> ICS Populated: 860 | NaPTAN Populated: 712

Validation Summary (JOINT vs. NaPTAN Prefix):
Validation_Check
Match                686
Missing NaPTAN       148
Review / Mismatch     16
Joint Hub             10
Name: count, dtype: int64


,STATION,JOINT,NaPTAN
59,Bethnal Green LU,TfL,910GBTHNLGR
135,Charing Cross LU,TfL,910GCHRNGXG
152,Chiswick,NR,940GZZLUCWP
197,Deptford,NR,940GZZDLDEP
251,Euston NR,NR,940GZZLUESQ
338,Heathrow Terminal 5 LU,TfL,910GHTRWTM5
341,Hendon,NR,940GZZLUHCL
372,Hounslow,NR,940GZZLUHWC
427,Lee,NR,940GZZLUMED
565,Putney,NR,940GZZLUEPY


## Network Edges: Adjacent Station Travel Times

### Data Source & Methodology:
* **Source:** Derived from TfL Line Route Sequence feeds and queried via the TfL Journey Planner API for active rail modes (Tube, DLR, Overground, Elizabeth Line and National Rail).

* **Assumptions & Fallbacks:** If a live API query fails or returns zero duration, a fallback travel time of `0.5` minutes is applied and logged to `travel_times_investigation_log.csv`.

In [ ]:
# ==============================================================================
# LINE TRAVEL TIMES BUILDER
# ==============================================================================
# Queries TfL Line Route sequences and Journey Planner API to compute 
# travel times between adjacent station pairs across all specified rail modes.
# ==============================================================================

RAIL_MODES = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
FALLBACK_TRAVEL_TIME = 0.5


def build_nlc_lookups():
  """
    Builds primary NLC lookup dictionaries from the loaded all_stations DataFrame and optional line-specific overrides.
  """
  nlc_map = {}
  clean_nlc_map = {}
  line_station_nlc_map = {}

  # 1. Load general station list from global 'all_stations' DataFrame
  try:
    for _, row in all_stations.iterrows():
      station_name = str(row["STATION"]).strip()
      nlc = row["NLC"]
      if pd.notna(nlc):
        nlc_map[station_name] = int(nlc)
        # Create cleaned lowercase lookup keys by stripping suffix labels
        clean_key = (
            station_name.replace(" DLR", "")
            .replace(" NR", "")
            .replace(" Station", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .strip()
            .lower()
        )
        clean_nlc_map[clean_key] = int(nlc)
    print(f"Loaded {len(nlc_map):,} NLC mappings from 'all_stations'.")

  except Exception as e:
    print(f"Warning/Error processing 'all_stations': {e}")

  # 2. Load missing NLC overrides (line-specific) if file exists
  try:
    if os.path.exists("missing NLC stations.csv"):
      missing_df = pd.read_csv("missing NLC stations.csv")
      for _, row in missing_df.iterrows():
        line = str(row["Line"]).strip()
        station = str(row["Station"]).strip()
        nlc = row["NLC"]
        if pd.notna(nlc):
          line_station_nlc_map[(line, station)] = int(nlc)
      print(
          f"Loaded {len(line_station_nlc_map)} line-specific overrides from"
          " 'missing NLC stations.csv'."
      )
  except Exception as e:
    print(f"Warning/Error loading 'missing NLC stations.csv': {e}")

  return nlc_map, clean_nlc_map, line_station_nlc_map


def lookup_nlc(
    line, station_name, nlc_map, clean_nlc_map, line_station_nlc_map
):
  """
    Helper to resolve NLC using line-specific overrides first, then exact station matches, then cleaned string matching.
  """
  if (line, station_name) in line_station_nlc_map:
    return line_station_nlc_map[(line, station_name)]

  if station_name in nlc_map:
    return nlc_map[station_name]

  clean_key = (
      station_name.replace(" DLR", "")
      .replace(" Underground Station", "")
      .replace(" Rail Station", "")
      .replace(" Station", "")
      .strip()
      .lower()
  )
  return clean_nlc_map.get(clean_key, "")


def get_line_travel_times():
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}

  nlc_map, clean_nlc_map, line_station_nlc_map = build_nlc_lookups()
  target_clean_names = set(clean_nlc_map.keys())

  # Step 1: Fetch all active lines across specified rail modes
  lines_url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(RAIL_MODES)}"
  res = requests.get(lines_url, params=params)
  if res.status_code != 200:
    print(f"Failed to fetch lines (Status {res.status_code}).")
    return pd.DataFrame()

  lines = res.json()
  records = []
  error_investigations = []

  print(f"Found {len(lines)} lines across modes. Processing station pairs...")

  for line in lines:
    line_id = line["id"]
    line_name = line["name"]

    print(f"Processing line: {line_name} ")

    # Step 2: Fetch route sequence branch stops for each line
    seq_url = f"https://api.tfl.gov.uk/Line/{line_id}/Route/Sequence/all"
    seq_res = requests.get(seq_url, params=params)
    if seq_res.status_code != 200:
      continue

    seq_data = seq_res.json()

    for branch in seq_data.get("stopPointSequences", []):
      stops = branch.get("stopPoint", [])

      # Step 3: Iterate through adjacent stop pairs along the branch
      for i in range(len(stops) - 1):
        from_stop, to_stop = stops[i], stops[i + 1]
        from_name, to_name = (
            from_stop.get("name", ""),
            to_stop.get("name", ""),
        )

        from_clean = (
            from_name.replace(" DLR", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .replace(" Station", "")
            .strip()
            .lower()
        )
        to_clean = (
            to_name.replace(" DLR", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .replace(" Station", "")
            .strip()
            .lower()
        )

        # Scope restriction: Skip segments where neither station is in our target network list
        if (
            from_clean not in target_clean_names
            and to_clean not in target_clean_names
        ):
          continue

        # Resolve NLC codes
        from_nlc = lookup_nlc(
            line_name, from_name, nlc_map, clean_nlc_map, line_station_nlc_map
        )
        to_nlc = lookup_nlc(
            line_name, to_name, nlc_map, clean_nlc_map, line_station_nlc_map
        )

        from_id, to_id = from_stop["id"], to_stop["id"]
        duration = None
        status_reason = "Success"

        # Step 4: Query TfL Journey Planner API for live scheduled travel duration
        jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(from_id)}/to/{urllib.parse.quote(to_id)}"

        try:
          jp_res = requests.get(jp_url, params=params, timeout=5)
          if jp_res.status_code == 200:
            journeys = jp_res.json().get("journeys", [])
            if journeys:
              legs = journeys[0].get("legs", [])
              if legs:
                raw_dur = legs[0].get("duration", journeys[0].get("duration"))
                if raw_dur is not None:
                  duration = float(raw_dur)
            else:
              status_reason = "No journeys returned in response"
          else:
            status_reason = f"HTTP Error {jp_res.status_code}"
        except Exception as err:
          status_reason = f"Request Exception: {err}"

        # Step 5: Handle missing or invalid durations via fallback rule
        if duration is None or duration <= 0:
          error_investigations.append({
              "Line": line_name,
              "From_NLC": from_nlc,
              "From_Station": from_name,
              "From_ID": from_id,
              "To_NLC": to_nlc,
              "To_Station": to_name,
              "To_ID": to_id,
              "Reason": status_reason,
              "Fallback_Used": FALLBACK_TRAVEL_TIME,
          })
          duration = FALLBACK_TRAVEL_TIME

        records.append({
            "Line": line_name,
            "From_NLC": from_nlc,
            "From_Station": from_name,
            "To_NLC": to_nlc,
            "To_Station": to_name,
            "Travel_Time_Mins": duration,
        })

        time.sleep(0.02)  # Rate limiting safeguard

  # Step 6: Format final DataFrame and export/return
  df = pd.DataFrame(records).drop_duplicates(
      subset=["Line", "From_Station", "To_Station"]
  )
  df["From_NLC"] = pd.to_numeric(df["From_NLC"], errors="coerce").astype("Int64")
  df["To_NLC"] = pd.to_numeric(df["To_NLC"], errors="coerce").astype("Int64")

  cols = [
      "Line",
      "From_NLC",
      "From_Station",
      "To_NLC",
      "To_Station",
      "Travel_Time_Mins",
  ]
  travel_times = df[cols]

  print(
      "\nSuccessfully generated adjacent station travel times with"
      f" {len(travel_times):,} station pairs."
  )

  if error_investigations:
    invest_df = pd.DataFrame(error_investigations)
    invest_df.to_csv("travel_times_investigation_log.csv", index=False)
    print(
        f"Logged {len(error_investigations):,} fallback travel times to"
        " 'travel_times_investigation_log.csv'."
    )

  return travel_times


travel_times = get_line_travel_times()
travel_times.head()

Loaded 852 NLC mappings from 'all_stations'.
Loaded 181 line-specific overrides from 'missing NLC stations.csv'.
Found 44 lines across modes. Processing station pairs...
Processing line: Avanti West Coast 


KeyboardInterrupt: 

## Network Edges: Adjacent Station Travel Times

### Data Source & Methodology:

* **Source:** Derived from TfL Line Route Sequence feeds (`/Line/{id}/Route/Sequence/all`) and queried via the TfL Journey Planner API (`/Journey/JourneyResults/{from_ics}/to/{to_ics}`) using official **ICS codes** mapped from `all_stations.csv`.

* **Assumptions & Fallbacks:** 
  * Only adjacent station pairs along active branch sequences are queried.
  * If a live API query fails or returns zero duration, a fallback travel time of `0.5` minutes (`FALLBACK_TRAVEL_TIME`) is applied and logged to `travel_times_investigation_log.csv`.

In [ ]:
# ==============================================================================
# LINE TRAVEL TIMES BUILDER (ICS-DRIVEN MAPPING & API QUERYING)
# ==============================================================================
# Maps route sequence stops to master NLCs and ICS station identifiers from 
# 'all_stations.csv', querying the TfL Journey Planner API using official ICS codes.
# ==============================================================================

RAIL_MODES = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
FALLBACK_TRAVEL_TIME = 0.5


def build_ics_and_nlc_lookups():
  """Builds robust lookup dictionaries mapping cleaned station names and API stop names

  directly to master NLC gate lines and ICS codes from the 'all_stations' DataFrame.
  """
  station_to_ics = {}
  station_to_nlc = {}
  clean_to_ics = {}
  clean_to_nlc = {}

  for _, row in all_stations.iterrows():
    station_name = str(row["STATION"]).strip()
    nlc = row["NLC"]
    ics = row["ICS"]

    if pd.notna(nlc) and pd.notna(ics):
      try:
        nlc_int = int(nlc)
        ics_str = str(int(float(ics)))

        station_to_ics[station_name] = ics_str
        station_to_nlc[station_name] = nlc_int

        # Cleaned normalization key for matching API sequence names
        clean_key = (
            station_name.replace(" DLR", "")
            .replace(" LU", "")
            .replace(" NR", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .replace(" Station", "")
            .strip()
            .lower()
        )
        clean_to_ics[clean_key] = ics_str
        clean_to_nlc[clean_key] = nlc_int
      except Exception:
        continue

  print(
      f"Loaded {len(station_to_ics):,} station mappings with ICS codes from"
      " 'all_stations'."
  )
  return station_to_ics, station_to_nlc, clean_to_ics, clean_to_nlc


def get_line_travel_times():
  # Standardize authentication using TFL_API_KEY
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}

  station_to_ics, station_to_nlc, clean_to_ics, clean_to_nlc = (
      build_ics_and_nlc_lookups()
  )
  target_clean_names = set(clean_to_ics.keys())

  # Step 1: Fetch all active lines across specified rail modes
  lines_url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(RAIL_MODES)}"
  res = requests.get(lines_url, params=params)
  if res.status_code != 200:
    print(f"Failed to fetch lines (Status {res.status_code}).")
    return pd.DataFrame()

  lines = res.json()
  records = []
  error_investigations = []

  print(f"Found {len(lines)} lines across modes. Processing station pairs...")

  for line in lines:
    line_id = line["id"]
    line_name = line["name"]

    # Step 2: Fetch route sequence branch stops for each line
    seq_url = f"https://api.tfl.gov.uk/Line/{line_id}/Route/Sequence/all"
    seq_res = requests.get(seq_url, params=params)
    if seq_res.status_code != 200:
      continue

    seq_data = seq_res.json()

    for branch in seq_data.get("stopPointSequences", []):
      stops = branch.get("stopPoint", [])

      # Step 3: Iterate through adjacent stop pairs along the branch
      for i in range(len(stops) - 1):
        from_stop, to_stop = stops[i], stops[i + 1]
        from_name, to_name = (
            from_stop.get("name", ""),
            to_stop.get("name", ""),
        )

        from_clean = (
            from_name.replace(" DLR", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .replace(" Station", "")
            .strip()
            .lower()
        )
        to_clean = (
            to_name.replace(" DLR", "")
            .replace(" Underground Station", "")
            .replace(" Rail Station", "")
            .replace(" Station", "")
            .strip()
            .lower()
        )

        # Scope restriction: Skip segments where neither station is in our target network list
        if (
            from_clean not in target_clean_names
            and to_clean not in target_clean_names
        ):
          continue

        # Resolve NLC and ICS codes
        from_nlc = clean_to_nlc.get(from_clean)
        to_nlc = clean_to_nlc.get(to_clean)
        from_ics = clean_to_ics.get(from_clean)
        to_ics = clean_to_ics.get(to_clean)

        if not from_ics or not to_ics:
          continue

        duration = None
        status_reason = "Success"

        # Step 4: Query TfL Journey Planner API using official ICS codes
        jp_url = f"https://api.tfl.gov.uk/Journey/JourneyResults/{urllib.parse.quote(from_ics)}/to/{urllib.parse.quote(to_ics)}"

        try:
          jp_res = requests.get(jp_url, params=params, timeout=5)
          if jp_res.status_code == 200:
            journeys = jp_res.json().get("journeys", [])
            if journeys:
              legs = journeys[0].get("legs", [])
              if legs:
                raw_dur = legs[0].get("duration", journeys[0].get("duration"))
                if raw_dur is not None:
                  duration = float(raw_dur)
            else:
              status_reason = "No journeys returned in response"
          else:
            status_reason = f"HTTP Error {jp_res.status_code}"
        except Exception as err:
          status_reason = f"Request Exception: {err}"

        # Step 5: Handle missing or invalid durations via fallback rule
        if duration is None or duration <= 0:
          error_investigations.append({
              "Line": line_name,
              "From_NLC": from_nlc,
              "From_Station": from_name,
              "From_ICS": from_ics,
              "To_NLC": to_nlc,
              "To_Station": to_name,
              "To_ICS": to_ics,
              "Reason": status_reason,
              "Fallback_Used": FALLBACK_TRAVEL_TIME,
          })
          duration = FALLBACK_TRAVEL_TIME

        records.append({
            "Line": line_name,
            "From_NLC": from_nlc,
            "From_Station": from_name,
            "To_NLC": to_nlc,
            "To_Station": to_name,
            "Travel_Time_Mins": duration,
        })

        time.sleep(0.02)  # Rate limiting safeguard

  # Step 6: Format final DataFrame and return
  df = pd.DataFrame(records).drop_duplicates(
      subset=["Line", "From_Station", "To_Station"]
  )
  df["From_NLC"] = pd.to_numeric(df["From_NLC"], errors="coerce").astype("Int64")
  df["To_NLC"] = pd.to_numeric(df["To_NLC"], errors="coerce").astype("Int64")

  cols = [
      "Line",
      "From_NLC",
      "From_Station",
      "To_NLC",
      "To_Station",
      "Travel_Time_Mins",
  ]
  travel_times = df[cols]

  print(
      "\nSuccessfully generated adjacent station travel times with"
      f" {len(travel_times):,} station pairs."
  )

  if error_investigations:
    invest_df = pd.DataFrame(error_investigations)
    invest_df.to_csv("travel_times_investigation_log.csv", index=False)
    print(
        f"Logged {len(error_investigations):,} fallback travel times to"
        " 'travel_times_investigation_log.csv'."
    )

  return travel_times


travel_times = get_line_travel_times()
travel_times.head()

Loaded 852 station mappings with ICS codes from 'all_stations'.
Found 44 lines across modes. Processing station pairs...

Successfully generated adjacent station travel times with 2,010 station pairs.
Logged 88 fallback travel times to 'travel_times_investigation_log.csv'.


,Line,From_NLC,From_Station,To_NLC,To_Station,Travel_Time_Mins
0,Bakerloo,597,Harrow & Wealdstone Underground Station,620,Kenton Underground Station,0.5
1,Bakerloo,620,Kenton Underground Station,709,South Kenton Underground Station,2.0
2,Bakerloo,709,South Kenton Underground Station,659,North Wembley Underground Station,2.0
3,Bakerloo,659,North Wembley Underground Station,751,Wembley Central Underground Station,0.5
4,Bakerloo,751,Wembley Central Underground Station,717,Stonebridge Park Underground Station,3.0


## Network Edges: Expected Line Wait Times

### Data Source & Methodology:
* **Source:** Extracted dynamically using the TfL Timetable API (`/Line/{id}/Timetable/{stop}`) at designated benchmark hub stations (`LINE_BENCHMARK_STATIONS`).

* **Assumptions & Fallbacks:** 
  * Passenger arrivals are assumed to be uniform and random.
  * Expected passenger wait time is computed using the half-headway rule: $\text{Wait} = \frac{60}{2 \times \text{TPH}}$ (Trains Per Hour), measuring peak arrivals at hour 08:00–09:00 and off-peak at 11:00–12:00.
  * Where timetable feeds are unavailable or fail, rule-based mode-level headways are applied as explicit fallbacks (e.g., Tube/DLR: 2.5m, Elizabeth Line: 2.0m, Overground: 5.0m, National Rail: 7.5m).

In [ ]:
# ==============================================================================
# LINE WAIT TIMES BUILDER & HALF-HEADWAY LOGIC
# ==============================================================================
# Queries TfL Timetable API for benchmark hub stations to calculate scheduled 
# Trains Per Hour (TPH) during peak and off-peak windows, falling back to 
# mode-level frequency rules where necessary.
# ==============================================================================

# Benchmark hub stations used to calculate line frequencies (TPH)
LINE_BENCHMARK_STATIONS = {
    "victoria": "940GZZLUOXC",  # Oxford Circus
    "jubilee": "940GZZLUBST",  # Baker Street
    "northern": "940GZZLUBNK",  # Bank
    "central": "940GZZLUOXC",  # Oxford Circus
    "bakerloo": "940GZZLUBST",  # Baker Street
    "piccadilly": "940GZZLUGPK",  # Green Park
    "district": "940GZZLUWMS",  # Westminster
    "circle": "940GZZLUBST",  # Baker Street
    "hammersmith-city": "940GZZLUBST", # Baker Street
    "metropolitan": "940GZZLUBST", # Baker Street
    "waterloo-city": "940GZZLUWLO",  # Waterloo
    "dlr": "940GZZDLBNK",  # Bank DLR
    "elizabeth": "940GZZLUTCR",  # Tottenham Court Road
}


def calculate_wait_from_tph(trains_per_hour: float) -> float:
  """Computes expected passenger wait time using the half-headway rule: Wait = 60 / (2 * TPH)."""
  if trains_per_hour <= 0:
    return 10.0
  return round(60.0 / (2.0 * trains_per_hour), 1)


def fetch_scheduled_tph_from_api(line_id: str, stop_naptan: str) -> dict:
  """Queries TfL Timetable API to calculate Peak and Off-Peak Trains Per Hour (TPH)."""
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  url = f"https://api.tfl.gov.uk/Line/{line_id}/Timetable/{stop_naptan}"

  try:
    res = requests.get(url, params=params, timeout=5)
    if res.status_code == 200:
      data = res.json()
      # Parse scheduled departures in peak (08:00-09:00) and off-peak (11:00-12:00)
      routes = data.get("timetable", {}).get("routes", [])
      peak_count = 0
      off_peak_count = 0

      for route in routes:
        for schedule in route.get("schedules", []):
          for entry in schedule.get("knownJourneys", []):
            hour = int(entry.get("hour", 0))
            if hour == 8:
              peak_count += 1
            elif hour == 11:
              off_peak_count += 1

      if peak_count > 0:
        tph_peak = max(2.0, float(peak_count))
        tph_offpeak = max(1.0, float(off_peak_count))
        return {
            "peak_wait": calculate_wait_from_tph(tph_peak),
            "offpeak_wait": calculate_wait_from_tph(tph_offpeak),
            "default_wait": calculate_wait_from_tph(
                (tph_peak + tph_offpeak) / 2.0
            ),
            "source": "TfL Timetable API (Scheduled TPH)",
        }
  except Exception:
    pass

  return None


def refresh_wait_times():
  """Generates a wait times table using dynamic API frequency extraction with published fallback rules."""
  params = {"app_key": TFL_API_KEY} if TFL_API_KEY else {}
  modes = ["tube", "dlr", "overground", "elizabeth-line", "national-rail"]
  url = f"https://api.tfl.gov.uk/Line/Mode/{','.join(modes)}"

  rows = []
  res = requests.get(url, params=params)

  if res.status_code == 200:
    lines_data = res.json()

    for item in lines_data:
      line_name = item.get("name", "").strip()
      mode = item.get("modeName", "").strip()
      line_id = item.get("id", "").strip().lower()

      # 1. Try dynamic API Timetable extraction
      naptan = LINE_BENCHMARK_STATIONS.get(line_id)
      api_metrics = (
          fetch_scheduled_tph_from_api(line_id, naptan) if naptan else None
      )

      if api_metrics:
        peak_w = api_metrics["peak_wait"]
        offpeak_w = api_metrics["offpeak_wait"]
        default_w = api_metrics["default_wait"]
        src = api_metrics["source"]
      else:
        # 2. Rule-based fallbacks (Mode-level average headways)
        if mode == "tube":
          peak_w, offpeak_w, default_w = 2.0, 3.5, 2.5
        elif mode == "dlr":
          peak_w, offpeak_w, default_w = 2.0, 3.5, 2.5
        elif mode == "elizabeth-line":
          peak_w, offpeak_w, default_w = 1.5, 2.5, 2.0
        elif mode == "overground":
          peak_w, offpeak_w, default_w = 4.0, 6.0, 5.0
        else:  # National Rail
          peak_w, offpeak_w, default_w = 5.0, 10.0, 7.5

        src = f"TfL Mode Frequency Rule ({mode.title()})"

      rows.append({
          "Line": line_name,
          "Mode": mode,
          "Peak Avg Wait (mins)": peak_w,
          "Off-Peak Avg Wait (mins)": offpeak_w,
          "Default Avg Wait (mins)": default_w,
          "Data_Source": src,
      })

  wait_times = (
      pd.DataFrame(rows).sort_values(by=["Mode", "Line"]).reset_index(drop=True)
  )

  print(f"Successfully generated {len(wait_times):,} line wait time profiles.")
  return wait_times


wait_times = refresh_wait_times()
wait_times.head()

Successfully generated 44 line wait time profiles.


,Line,Mode,Peak Avg Wait (mins),Off-Peak Avg Wait (mins),Default Avg Wait (mins),Data_Source
0,DLR,dlr,0.6,0.6,0.6,TfL Timetable API (Scheduled TPH)
1,Elizabeth line,elizabeth-line,1.5,2.5,2.0,TfL Mode Frequency Rule (Elizabeth-Line)
2,Avanti West Coast,national-rail,5.0,10.0,7.5,TfL Mode Frequency Rule (National-Rail)
3,Chiltern Railways,national-rail,5.0,10.0,7.5,TfL Mode Frequency Rule (National-Rail)
4,CrossCountry,national-rail,5.0,10.0,7.5,TfL Mode Frequency Rule (National-Rail)


## Network Edges: Interchange Walking Penalties

### Data Source & Methodology:
* **Source:** Combines published TfL Pay As You Go Out-of-Station Interchange (OSI) timeout matrix (`osi_matrix.csv`) with synthetic API journey triads for in-station platform transfers.

* **Assumptions & Fallbacks:** 
  * OSI fare timeout limits are mapped to physical walking durations ($\le 10\text{m} \to 4.0\text{m}$, $\le 15\text{m} \to 5.0\text{m}$, etc.).

  * **In-Station Interchanges:** Evaluated using a 4-tier hierarchy:
    1. *Empirical Layout Overrides* (e.g., Bank, King's Cross, Waterloo complex hubs).
    2. *Cross-Platform Benchmarks* (2.0m flat penalty for same-platform transfers).
    3. *Synthetic API Triads* (${\text{Walk} = \text{Journey}_{A \to C} - (\text{Leg}_A + \text{Leg}_B + \text{Wait}_B)}$).
    4. *Depth-Tier Heuristics* (Fallback rule based on surface, subsurface, and deep tube layering).

  * **OSI Exclusion:** Out-of-station interchanges will be skipped when building the strict in-station routing graph.

In [ ]:
# ==============================================================================
# SYSTEMATIC INTERCHANGE WALTING PENALTIES & CLASSIFICATION BUILDER
# ==============================================================================
# Categorizes station transfers into Behind-the-Gate In-Station Interchanges, 
# Gated OSIs, and Ungated OSIs using NLC counts and GATED status from all_stations.csv.
# ==============================================================================


def convert_osi_timeout_to_walk_mins(timeout_mins: float) -> float:
  """Converts TfL PAYG revenue timeout limits into physical walking durations."""
  timeout_mins = float(timeout_mins)
  if timeout_mins <= 10:
    return 4.0
  elif timeout_mins <= 15:
    return 5.0
  elif timeout_mins <= 20:
    return 7.0
  elif timeout_mins <= 30:
    return 8.5
  else:
    return 10.0


# Depth tier fallbacks for physical platform walking durations
LINE_DEPTH_TIERS = {
    "Surface/Rail": {
        "National Rail",
        "London Overground",
        "c2c",
        "Chiltern Railways",
        "Southeastern",
        "Thameslink",
        "DLR",
        "Great Northern",
        "Great Western Railway",
        "South Western Railway",
        "Lioness",
        "Mildmay",
        "Windrush",
        "Weaver",
        "Suffragette",
        "Liberty",
    },
    "Subsurface": {"Circle", "District", "Hammersmith & City", "Metropolitan"},
    "Deep Tube": {
        "Bakerloo",
        "Central",
        "Jubilee",
        "Northern",
        "Piccadilly",
        "Victoria",
        "Waterloo & City",
        "Elizabeth line",
    },
}


def get_line_tier(line_name: str) -> str:
  for tier, lines in LINE_DEPTH_TIERS.items():
    if line_name in lines:
      return tier
  return "Deep Tube"


def get_depth_tier_walk_mins(line_a: str, line_b: str) -> float:
  t_a, t_b = get_line_tier(line_a), get_line_tier(line_b)
  if t_a == t_b:
    return 2.0
  elif {t_a, t_b} == {"Surface/Rail", "Deep Tube"}:
    return 7.0
  elif {t_a, t_b} == {"Subsurface", "Deep Tube"}:
    return 5.0
  elif {t_a, t_b} == {"Surface/Rail", "Subsurface"}:
    return 4.0
  return 3.0


# Strict behind-the-gate DLR multi-line hub overrides (where distinct lines share NLCs)
STRICT_DLR_IN_STATION_HUBS = {
    513,  # Bank / Monument Unified
    719,  # Stratford
    884,  # Canning Town
    5145,  # Greenwich
}


def generate_systematic_interchanges():
  df_stations = all_stations.copy()
  df_stations["NLC_int"] = (
      pd.to_numeric(df_stations["NLC"], errors="coerce")
      .dropna()
      .astype(int)
  )

  # Station name NLC multiplicity map
  station_nlc_counts = df_stations.groupby("STATION")["NLC_int"].nunique().to_dict()

  # NLC Gated status map ("gated" vs "ungated")
  nlc_gated_map = dict(
      zip(
          df_stations["NLC_int"],
          df_stations["GATED"].str.strip().str.lower().fillna("gated"),
      )
  )

  nlc_name_map = dict(
      zip(df_stations["NLC_int"], df_stations["STATION"].str.strip())
  )

  df_travel = travel_times
  df_wait = wait_times
  df_osi = pd.read_csv("osi_matrix.csv")

  wait_dict = dict(
      zip(
          df_wait["Line"].str.strip(),
          df_wait["Default Avg Wait (mins)"].astype(float),
      )
  )

  records = []

  # STEP 1: Process Published Out-of-Station Interchanges (OSIs) from Report
  for _, row in df_osi.iterrows():
    try:
      nlc_a = int(row["Station_A"])
      nlc_b = int(row["Station_B"])

      if nlc_b == 0 or nlc_a == nlc_b:
        continue

      raw_timeout = float(row["InterchangeTimeAB"])
      walk_mins = convert_osi_timeout_to_walk_mins(raw_timeout)

      st_a = nlc_name_map.get(nlc_a, str(row["Station_A_Name"]).strip())
      st_b = nlc_name_map.get(nlc_b, str(row["Station_B_Name"]).strip())
      combined_name = f"{st_a} / {st_b}"

      lines_a = df_travel[
          (df_travel["From_NLC"] == nlc_a) | (df_travel["To_NLC"] == nlc_a)
      ]["Line"].unique()
      lines_b = df_travel[
          (df_travel["From_NLC"] == nlc_b) | (df_travel["To_NLC"] == nlc_b)
      ]["Line"].unique()

      gated_a = nlc_gated_map.get(nlc_a, "gated")
      gated_b = nlc_gated_map.get(nlc_b, "gated")
      ic_type = (
          "Out-of-Station Interchange (OSI)"
          if (gated_a == "gated" and gated_b == "gated")
          else "Ungated Out-of-Station Interchange"
      )

      for l_a in lines_a:
        for l_b in lines_b:
          records.append({
              "station": combined_name,
              "station_NLC": nlc_a,
              "From_Line": l_a.strip(),
              "To_Line": l_b.strip(),
              "walk_interchange_times": walk_mins,
              "interchange_type": ic_type,
              "source": "Published TfL OSI Matrix",
          })
    except Exception:
      continue

  # STEP 2: Systematic In-Station vs OSI Classification for Multi-Line Hubs
  from_df = df_travel[["From_NLC", "From_Station", "Line"]].rename(
      columns={"From_NLC": "NLC", "From_Station": "Station"}
  )
  to_df = df_travel[["To_NLC", "To_Station", "Line"]].rename(
      columns={"To_NLC": "NLC", "To_Station": "Station"}
  )
  station_lines = pd.concat([from_df, to_df]).drop_duplicates()

  multi_line_hubs = (
      station_lines.groupby("NLC")["Line"].nunique().loc[lambda x: x > 1].index
  )

  for nlc in multi_line_hubs:
    sub = station_lines[station_lines["NLC"] == nlc]
    st_name = sub["Station"].iloc[0].strip()
    lines = sub["Line"].tolist()
    unique_lines = list(set([l.strip() for l in lines]))

    mapped_nlc = NLC_ALIASES.get(int(nlc), int(nlc))
    num_nlcs_at_station = station_nlc_counts.get(st_name, 1)
    gated_status = nlc_gated_map.get(int(nlc), "gated")

    for i in range(len(unique_lines)):
      for j in range(i + 1, len(unique_lines)):
        l_a, l_b = unique_lines[i], unique_lines[j]

        # Systematic Classification Rule:
        # If single NLC station OR whitelisted behind-the-gate hub -> In-Station Interchange
        # Otherwise -> Out-of-Station Interchange (OSI) or Concourse Transfer
        is_dlr_involved = (
            "dlr" in l_a.lower()
            or "dlr" in l_b.lower()
            or "dlr" in st_name.lower()
        )

        if num_nlcs_at_station == 1 or mapped_nlc in STRICT_DLR_IN_STATION_HUBS:
          ic_type = "In-Station Interchange"
          source = "Systematic Rule (Single NLC / Gated Hub)"
          walk_mins = 2.0  # Default flat cross-platform walk
        elif gated_status == "ungated":
          ic_type = "Ungated Out-of-Station Interchange"
          source = "Systematic Rule (Ungated Node)"
          walk_mins = 4.0
        else:
          ic_type = "Out-of-Station Interchange (OSI)"
          source = "Systematic Rule (Multi-NLC Barrier Separation)"
          walk_mins = get_depth_tier_walk_mins(l_a, l_b)

        records.append({
            "station": st_name,
            "station_NLC": int(nlc),
            "From_Line": l_a,
            "To_Line": l_b,
            "walk_interchange_times": walk_mins,
            "interchange_type": ic_type,
            "source": source,
        })

  # STEP 3: Deduplicate & Enforce Bidirectional Symmetry
  df_out = pd.DataFrame(records).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  reverse_records = []
  for _, row in df_out.iterrows():
    reverse_records.append({
        "station": row["station"],
        "station_NLC": row["station_NLC"],
        "From_Line": row["To_Line"],
        "To_Line": row["From_Line"],
        "walk_interchange_times": row["walk_interchange_times"],
        "interchange_type": row["interchange_type"],
        "source": row["source"],
    })

  df_final = pd.concat([df_out, pd.DataFrame(reverse_records)]).drop_duplicates(
      subset=["station_NLC", "From_Line", "To_Line"]
  )

  cols = [
      "station",
      "station_NLC",
      "From_Line",
      "To_Line",
      "walk_interchange_times",
      "interchange_type",
      "source",
  ]
  interchange_times = df_final[cols].sort_values(
      by=["station", "From_Line", "To_Line"]
  )

  print(
      "Successfully generated"
      f" {len(interchange_times):,} systematically classified interchange"
      " edges."
  )
  print("\nBreakdown by Interchange Type:")
  print(interchange_times["interchange_type"].value_counts())

  return interchange_times

interchange_times = generate_systematic_interchanges()
interchange_times.head()

Successfully generated 1,042 systematically classified interchange edges.

Breakdown by Interchange Type:
interchange_type
In-Station Interchange                829
Out-of-Station Interchange (OSI)      163
Ungated Out-of-Station Interchange     50
Name: count, dtype: int64


,station,station_NLC,From_Line,To_Line,walk_interchange_times,interchange_type,source
320,Acton Main Line Rail Station,3000,Elizabeth line,Great Western Railway,2.0,In-Station Interchange,Systematic Rule (Single NLC / Gated Hub)
349,Acton Main Line Rail Station,3000,Great Western Railway,Elizabeth line,2.0,In-Station Interchange,Systematic Rule (Single NLC / Gated Hub)
136,Acton Town Underground Station,500,District,Piccadilly,2.0,In-Station Interchange,Systematic Rule (Single NLC / Gated Hub)
114,Acton Town Underground Station,500,Piccadilly,District,2.0,In-Station Interchange,Systematic Rule (Single NLC / Gated Hub)
0,Aldgate / Aldgate East,502,Circle,District,4.0,Out-of-Station Interchange (OSI),Published TfL OSI Matrix


## Network Graph Construction (`NetworkX`) with NLC Aliasing

Constructs a directed graph representing the London contactless transit network:

* **NLC Aliasing:** Harmonizes legacy, variant, and adjacent station codes (e.g., Monument $\to$ Bank, Bow Road $\to$ Bow Church) using `NLC_ALIASES` to ensure methodological consistency.

* **Nodes:** Represent unique physical gate lines (`NLC_Line`), keeping multi-modal platforms (London Bridge NR vs. Underground) separate.

* **Edges (OSIs Excluded):** Journey travel times, boarding wait times, and in-station interchange walking penalties.

In [ ]:
# ==============================================================================
# NETWORK GRAPH CONSTRUCTION & DIJKSTRA TOPOLOGY BUILDER (WITH NLC ALIASES)
# ==============================================================================
# Constructs a directed NetworkX graph with NLC aliasing applied to nodes and edges.
# Out-of-Station Interchanges (OSIs) are strictly excluded.
# ==============================================================================

print("Initializing NetworkX Directed Graph...")
G = nx.DiGraph()

# Mapping of legacy/variant NLCs and adjacent station stops to primary DLR/hub NLCs
NLC_ALIASES = {
    # Legacy / Variant DLC & Rail NLC Unification
    843: 884,  # Canning Town DLR -> Canning Town Primary NLC
    573: 5152,  # Woolwich Arsenal DLR -> Woolwich Arsenal Primary NLC
    844: 513,  # Bank DLR -> Bank Primary NLC
    863: 719,  # Stratford DLR -> Stratford Primary NLC
    6960: 719,  # Stratford NR -> Stratford Primary NLC
    1082: 860,  # Shadwell LO -> Shadwell Primary NLC
    842: 852,  # Canary Wharf DLR -> Canary Wharf Primary NLC
    887: 6561,  # Custom House DLR -> Custom House Primary NLC
    928: 5145,  # Greenwich DLR -> Greenwich Primary NLC
    862: 7491,  # Limehouse DLR -> Limehouse Primary NLC
    940: 5051,  # Lewisham DLR -> Lewisham Primary NLC
    559: 889,  # West Silvertown DLR -> West Silvertown Primary NLC

    # Adjacent Station / Cross-Platform Mappings (Consistency with Manual Review)
    528: 851,  # Bow Road LU -> Bow Church DLR
    644: 513,  # Monument LU -> Bank Unified Hub
    731: 888,  # Tower Hill LU -> Tower Gateway DLR
}

# 1. Build lookup maps and apply NLC_ALIASES to station NLCs
df_stations = all_stations.copy()
df_stations["NLC_int"] = pd.to_numeric(
    df_stations["NLC"], errors="coerce"
).astype("Int64")

# Map original NLC to aliased primary NLC
nlc_to_name = {}
for _, row in df_stations.dropna(subset=["NLC_int"]).iterrows():
  orig_nlc = int(row["NLC_int"])
  mapped_nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)
  station_name = str(row["STATION"]).strip()
  nlc_to_name[mapped_nlc] = station_name

# Identify DLR stations using mapped NLCs
dlr_stations_df = df_stations[
    df_stations["STATION"].str.contains("DLR", case=False, na=False)
]
dlr_nlcs = set(
    dlr_stations_df["NLC_int"]
    .dropna()
    .astype(int)
    .map(lambda x: NLC_ALIASES.get(x, x))
)

print(
    f"Loaded {len(nlc_to_name):,} unique station nodes ({len(dlr_nlcs)} DLR"
    " endpoint groups)."
)

# 2. Add Journey Edges (with NLC Aliasing applied)
travel_added = 0
for _, row in travel_times.iterrows():
  try:
    orig_from_nlc = int(row["From_NLC"])
    orig_to_nlc = int(row["To_NLC"])

    # Apply NLC Aliasing
    from_nlc = NLC_ALIASES.get(orig_from_nlc, orig_from_nlc)
    to_nlc = NLC_ALIASES.get(orig_to_nlc, orig_to_nlc)

    line = str(row["Line"]).strip()
    duration = float(row["Travel_Time_Mins"])

    suffix = "DLR" if line.lower() == "dlr" else line
    u_node = f"{from_nlc}_{suffix}"
    v_node = f"{to_nlc}_{suffix}"

    G.add_edge(u_node, v_node, weight=duration, edge_type="journey", line=line)
    G.add_edge(v_node, u_node, weight=duration, edge_type="journey", line=line)
    travel_added += 2
  except Exception:
    continue

print(f"Added {travel_added:,} directional journey edges.")

# 3. Add Wait Edges (Initial boarding wait times)
wait_dict = dict(
    zip(
        wait_times["Line"].str.strip(),
        wait_times["Default Avg Wait (mins)"].astype(float),
    )
)
wait_added = 0
for nlc in nlc_to_name.keys():
  station_lines = travel_times[
      (travel_times["From_NLC"].map(lambda x: NLC_ALIASES.get(int(x), int(x)))
       == nlc)
      | (
          travel_times["To_NLC"].map(
              lambda x: NLC_ALIASES.get(int(x), int(x))
          )
          == nlc
      )
  ]["Line"].unique()

  for line in station_lines:
    suffix = "DLR" if str(line).lower() == "dlr" else str(line)
    node_id = f"{nlc}_{suffix}"
    if G.has_node(node_id):
      wait_time = wait_dict.get(str(line).strip(), 2.5)
      G.add_edge(
          node_id, node_id, weight=wait_time, edge_type="wait", line=line
      )
      wait_added += 1

print(f"Added {wait_added:,} station boarding wait edges.")

# 4. Add Interchange Edges (In-station platform transfers, OSIs strictly excluded)
interchange_added = 0
for _, row in interchange_times.iterrows():
  try:
    if "OSI" in str(row["interchange_type"]):
      continue  # Skip OSIs

    orig_nlc = int(row["station_NLC"])
    nlc = NLC_ALIASES.get(orig_nlc, orig_nlc)

    l_from = str(row["From_Line"]).strip()
    l_to = str(row["To_Line"]).strip()
    walk_time = float(row["walk_interchange_times"])

    s_from = "DLR" if l_from.lower() == "dlr" else l_from
    s_to = "DLR" if l_to.lower() == "dlr" else l_to

    u_node = f"{nlc}_{s_from}"
    v_node = f"{nlc}_{s_to}"

    if G.has_node(u_node) and G.has_node(v_node):
      G.add_edge(
          u_node,
          v_node,
          weight=walk_time,
          edge_type="interchange",
          line=f"{l_from}->{l_to}",
      )
      interchange_added += 1
  except Exception:
    continue

print(
    "Added"
    f" {interchange_added:,} in-station interchange edges (OSIs excluded)."
)
print(
    "\nGraph Construction Complete: Total Nodes ="
    f" {len(G.nodes):,} | Total Edges = {len(G.edges):,}"
)

Initializing NetworkX Directed Graph...
Loaded 848 unique station nodes (9 DLR endpoint groups).
Added 4,020 directional journey edges.
Added 1,047 station boarding wait edges.
Added 841 in-station interchange edges (OSIs excluded).

Graph Construction Complete: Total Nodes = 1,047 | Total Edges = 3,919


## Serialize & Save Network Graph

Saves the in-memory NetworkX directed graph and associated lookup dictionaries (`nlc_to_name`, `df_stations`) to disk (`network_graph_no_osi.pkl`). This allows subsequent routing and auditing cells to load the network topology instantly without rebuilding it.

In [ ]:
# ==============================================================================
# SERIALIZE & SAVE NETWORK GRAPH
# ==============================================================================
# Packages the NetworkX graph and lookup metadata into a dictionary and 
# serializes it using pickle for downstream routing scripts and notebooks.
# ==============================================================================

# Bundle graph and metadata structures
graph_data = {
    "graph": G,
    "nlc_to_name": nlc_to_name,
    "stations": df_stations,
}

output_filename = "network_graph.pkl"

with open(output_filename, "wb") as f:
  pickle.dump(graph_data, f)

print(
    f"Successfully serialized and saved network graph to '{output_filename}'"
    f" ({os.path.getsize(output_filename) / (1024 * 1024):.2f} MB)."
)

Successfully serialized and saved network graph to 'network_graph.pkl' (0.36 MB).


## DLR Routing Engine

Executes Dijkstra shortest-path routing across the network graph from all non-DLR station gate lines to all DLR station endpoints (and vice versa):

* **In-Gate Routing Only:** OSIs are excluded, ensuring only valid platform-to-platform transfers are used.

* **DLR Interchange Identification:** Traces each shortest path to identify the exact node where the journey transitions onto or off the DLR network.

* **Output Schema:** Produces a comprehensive CSV table (`dlr_journey_interchanges.csv`) containing origin/destination names, NLCs, designated DLR interchanges, full interchange station lists, and formatted journey times.

In [ ]:
# ==============================================================================
# BATCH DLR ROUTING ENGINE & INTERCHANGE AUDIT BUILDER
# ==============================================================================
# Computes shortest paths between all non-DLR and DLR NLC gate lines,
# extracts transfer transition points, and formats the output into the 
# required 8-column schema.
# ==============================================================================


def get_route_interchanges_and_dlr_point(path, nlc_to_name):
  """Traces a NetworkX path to identify all interchange stations

  and the specific DLR interchange transition point.
  """
  dlr_interchange_nlc = None
  dlr_interchange_name = None
  all_interchanges = []

  for i in range(len(path) - 1):
    u_node, v_node = path[i], path[i + 1]
    u_parts = u_node.split("_")
    v_parts = v_node.split("_")

    u_nlc, u_line = int(u_parts[0]), u_parts[1]
    v_nlc, v_line = int(v_parts[0]), v_parts[1]

    # Detect line transition (Interchange)
    if u_line != v_line and u_nlc == v_nlc:
      mapped_nlc = NLC_ALIASES.get(u_nlc, u_nlc)
      st_name = nlc_to_name.get(mapped_nlc, f"NLC_{mapped_nlc}")
      if st_name not in all_interchanges:
        all_interchanges.append(st_name)

    # Detect DLR transition (Onto or off DLR)
    if (u_line == "DLR" and v_line != "DLR") or (
        u_line != "DLR" and v_line == "DLR"
    ):
      transition_nlc = u_nlc if u_line == "DLR" else v_nlc
      mapped_trans_nlc = NLC_ALIASES.get(transition_nlc, transition_nlc)
      dlr_interchange_nlc = mapped_trans_nlc
      dlr_interchange_name = nlc_to_name.get(
          mapped_trans_nlc, f"NLC_{mapped_trans_nlc}"
      )

  # If route is entirely DLR or direct without explicit boundary node capture
  if not dlr_interchange_name and path:
    for node in path:
      parts = node.split("_")
      if parts[1] == "DLR":
        raw_nlc = int(parts[0])
        mapped_nlc = NLC_ALIASES.get(raw_nlc, raw_nlc)
        dlr_interchange_nlc = mapped_nlc
        dlr_interchange_name = nlc_to_name.get(
            mapped_nlc, f"NLC_{mapped_nlc}"
        )
        break

  all_interchanges_str = (
      ", ".join(all_interchanges) if all_interchanges else "Direct / None"
  )
  return dlr_interchange_nlc, dlr_interchange_name, all_interchanges_str


def run_batch_dlr_routing():
  # Identify DLR nodes and NLCs
  dlr_nodes = [n for n in G.nodes if n.endswith("_DLR")]
  dlr_nlcs = set(int(n.split("_")[0]) for n in dlr_nodes)
  all_nlcs = set(df_stations["NLC"].dropna().astype(int))
  non_dlr_nlcs = all_nlcs - dlr_nlcs

  print(
      f"Running batch routing across {len(dlr_nlcs)} DLR endpoint NLCs and"
      f" {len(non_dlr_nlcs)} non-DLR NLC gate lines..."
  )
  results = []

  # Helper to process and append route results
  def add_route_record(direction, u_nlc, v_nlc, cost, path):
    total_mins = int(round(cost))
    hours, mins = divmod(total_mins, 60)
    journey_time_str = f"{hours}:{mins:02d}"

    u_mapped = NLC_ALIASES.get(u_nlc, u_nlc)
    v_mapped = NLC_ALIASES.get(v_nlc, v_nlc)

    from_station = nlc_to_name.get(u_mapped, f"NLC_{u_mapped}")
    to_station = nlc_to_name.get(v_mapped, f"NLC_{v_mapped}")

    ic_nlc, ic_name, all_ics = get_route_interchanges_and_dlr_point(
        path, nlc_to_name
    )

    results.append({
        "from station": from_station,
        "from station NLC": u_mapped,
        "to station": to_station,
        "to station NLC": v_mapped,
        "dlr interchange": ic_name,
        "dlr interchange NLC": ic_nlc,
        "all interchange stations": all_ics,
        "journey time": journey_time_str,
        "Journey_Time_Mins": total_mins,
        "Direction": direction,
    })

  # 1. Non-DLR -> DLR Journeys
  for u_nlc in non_dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_")]
    if not u_nodes:
      continue
    for v_nlc in dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_DLR")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "Non-DLR to DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # 2. DLR -> Non-DLR Journeys
  for u_nlc in dlr_nlcs:
    u_nodes = [n for n in G.nodes if n.startswith(f"{u_nlc}_DLR")]
    if not u_nodes:
      continue
    for v_nlc in non_dlr_nlcs:
      v_nodes = [n for n in G.nodes if n.startswith(f"{v_nlc}_")]
      if not v_nodes:
        continue

      best_cost, best_path = float("inf"), None
      for s in u_nodes:
        for t in v_nodes:
          if nx.has_path(G, s, t):
            c = nx.shortest_path_length(G, s, t, weight="weight")
            if c < best_cost:
              best_cost, best_path = c, nx.shortest_path(
                  G, s, t, weight="weight"
              )

      if best_path:
        add_route_record(
            "DLR to Non-DLR", u_nlc, v_nlc, best_cost, best_path
        )

  # Format output DataFrame
  df_out = pd.DataFrame(results).drop_duplicates(
      subset=["from station NLC", "to station NLC"]
  )

  cols = [
      "from station",
      "from station NLC",
      "to station",
      "to station NLC",
      "dlr interchange",
      "dlr interchange NLC",
      "all interchange stations",
      "journey time",
  ]
  master_routes_df = df_out[cols]

  master_routes_df.to_csv("dlr_journey_interchanges.csv", index=False)
  print(
      "\nRouting Complete: Successfully generated"
      f" {len(master_routes_df):,} total route pairs in"
      " 'dlr_journey_interchanges.csv'."
  )
  return master_routes_df

master_routes_df = run_batch_dlr_routing()
master_routes_df.head()

Running batch routing across 42 DLR endpoint NLCs and 818 non-DLR NLC gate lines...

Routing Complete: Successfully generated 55,146 total route pairs in 'dlr_journey_interchanges.csv'.


,from station,from station NLC,to station,to station NLC,dlr interchange,dlr interchange NLC,all interchange stations,journey time
0,Welwyn Garden City,6183,Deptford Bridge,897,Greenwich,5145,"Knebworth, Alexandra Palace, Harringay, Finsbu...",1:09
1,Welwyn Garden City,6183,Elverson Road,898,Greenwich,5145,"Knebworth, Alexandra Palace, Harringay, Finsbu...",1:11
2,Welwyn Garden City,6183,Monument,513,Monument,513,"Knebworth, Alexandra Palace, Harringay, Finsbu...",1:00
3,Welwyn Garden City,6183,King George V,523,Woolwich Arsenal NR,5152,"Knebworth, Alexandra Palace, Harringay, Finsbu...",1:24
4,Welwyn Garden City,6183,London City Airport,533,Woolwich Arsenal NR,5152,"Knebworth, Alexandra Palace, Harringay, Finsbu...",1:26


In [ ]:
# ==============================================================================
# DLR ROUTING AUDIT & COMPARISON SCRIPT
# ==============================================================================
# Compares 'dlr_journey_interchanges.csv' against the manual review baseline.
# ==============================================================================

import os
import pandas as pd

# 1. Load datasets
manual_file = "DLR interchange manual review.csv"
auto_file = "dlr_journey_interchanges.csv"

if not os.path.exists(manual_file) or not os.path.exists(auto_file):
  print(
      "Error: Required CSV files not found. Ensure both files are in the"
      " directory."
  )
else:
  df_manual = pd.read_csv(manual_file)
  df_auto = pd.read_csv(auto_file)

  # Standardize column names
  df_manual.columns = df_manual.columns.str.strip()
  df_auto.columns = df_auto.columns.str.strip()

  # Master NLC Alias Unification Map (to bridge old/new and adjacent stations)
  NLC_ALIASES = {
      843: 884,  # Canning Town
      573: 5152,  # Woolwich Arsenal
      844: 513,  # Bank
      863: 719,  # Stratford
      6960: 719,  # Stratford NR
      1082: 860,  # Shadwell LO
      842: 852,  # Canary Wharf
      887: 6561,  # Custom House
      928: 5145,  # Greenwich
      862: 7491,  # Limehouse
      940: 5051,  # Lewisham
      559: 889,  # West Silvertown
      528: 851,  # Bow Road LU -> Bow Church DLR
      644: 513,  # Monument LU -> Bank Unified Hub
      731: 888,  # Tower Hill LU -> Tower Gateway DLR
  }

  # Apply NLC mapping
  df_manual["NLC1_mapped"] = (
      pd.to_numeric(df_manual["NLC1"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_manual["NLC2_mapped"] = (
      pd.to_numeric(df_manual["NLC2"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  df_auto["from_NLC_mapped"] = (
      pd.to_numeric(df_auto["from station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )
  df_auto["to_NLC_mapped"] = (
      pd.to_numeric(df_auto["to station NLC"], errors="coerce")
      .fillna(0)
      .astype(int)
      .replace(NLC_ALIASES)
  )

  # Deduplicate
  m_uniq = df_manual.drop_duplicates(subset=["NLC1_mapped", "NLC2_mapped"]).copy()
  a_uniq = df_auto.drop_duplicates(
      subset=["from_NLC_mapped", "to_NLC_mapped"]
  ).copy()

  # Metric 1: Total number of routes and unique NLC stations
  print("=== 1. Total Routes & Unique NLC Stations ===")
  print(f"Total Unique Routes in Manual Review: {len(m_uniq):,}")
  print(f"Total Unique Routes in Automated Graph: {len(a_uniq):,}")

  manual_nlcs = set(m_uniq["NLC1_mapped"]).union(set(m_uniq["NLC2_mapped"]))
  auto_nlcs = set(a_uniq["from_NLC_mapped"]).union(
      set(a_uniq["to_NLC_mapped"])
  )
  print(f"Unique NLC Gate Lines in Manual Review: {len(manual_nlcs):,}")
  print(f"Unique NLC Gate Lines in Automated Graph: {len(auto_nlcs):,}")

  # Metric 2: Distribution of DLR Interchanges Found
  print("\n=== 2. Distribution of DLR Interchanges Found (Top Hubs) ===")
  print("\nManual Review DLR Interchanges:")
  print(m_uniq["DLR_INT Name"].value_counts().head(10))

  print("\nAutomated Graph DLR Interchanges:")
  print(a_uniq["dlr interchange"].value_counts().head(10))

  # Metric 3: Routes with no interchanges vs interchanges present
  m_no_ic = m_uniq["DLR_INT"].isna() | (m_uniq["DLR_INT"] == 0)
  a_no_ic = (
      a_uniq["dlr interchange"].isna()
      | (a_uniq["dlr interchange"] == "Direct / None")
      | (a_uniq["dlr interchange"] == "")
  )

  print("\n=== 3. Interchange Presence Comparison ===")
  print(f"Manual Routes with No DLR Interchange Logged: {m_no_ic.sum():,}")
  print(f"Automated Routes with No DLR Interchange Found: {a_no_ic.sum():,}")

  # Merge datasets for direct overlap analysis
  merged = pd.merge(
      m_uniq,
      a_uniq,
      left_on=["NLC1_mapped", "NLC2_mapped"],
      right_on=["from_NLC_mapped", "to_NLC_mapped"],
      how="outer",
      indicator=True,
  )

  print("\n=== 4. Dataset Overlap & Agreement ===")
  merge_counts = merged["_merge"].value_counts()
  print(f"Shared O-D Pairs (Matched Both): {merge_counts.get('both', 0):,}")
  print(f"Manual Review Only: {merge_counts.get('left_only', 0):,}")
  print(f"Automated Graph Only: {merge_counts.get('right_only', 0):,}")

=== 1. Total Routes & Unique NLC Stations ===
Total Unique Routes in Manual Review: 66,298
Total Unique Routes in Automated Graph: 55,146
Unique NLC Gate Lines in Manual Review: 790
Unique NLC Gate Lines in Automated Graph: 699

=== 2. Distribution of DLR Interchanges Found (Top Hubs) ===

Manual Review DLR Interchanges:
DLR_INT Name
Bank                    39040
Stratford                8548
Lewisham DLR             6293
Canary Wharf DLR         3113
Canning Town             2234
Limehouse DLR            1833
Bow Church               1301
Shadwell DLR              617
Woolwich Arsenal DLR      430
West Ham                  373
Name: count, dtype: int64

Automated Graph DLR Interchanges:
dlr interchange
Canary Wharf LU (E1)    15279
Tower Hill               7746
Stratford                6581
Woolwich Arsenal NR      5128
Greenwich                4152
Monument                 3758
Shadwell LO              3231
Canning Town             1921
Bow Road                 1809
Limehouse NR     

In [ ]:
# Diagnostic: Find NLCs present in manual review but missing from automated graph
manual_nlcs = set(df_manual['NLC1']).union(set(df_manual['NLC2']))
auto_nlcs = set(df_auto['from station NLC']).union(
    set(df_auto['to station NLC'])
)

missing_nlcs = manual_nlcs - auto_nlcs
print(f'Number of NLCs in manual review missing from automated graph: {len(missing_nlcs)}')

Number of NLCs in manual review missing from automated graph: 155


In [ ]:
# ==============================================================================
# DIAGNOSTIC: FIND STATIONS IN all_stations.csv MISSING FROM line_travel_times.csv
# ==============================================================================

# 1. Load datasets
df_stations = pd.read_csv("all_stations.csv")
df_travel = pd.read_csv("line_travel_times.csv")

# 2. Extract unique NLC sets
all_station_nlcs = set(
    pd.to_numeric(df_stations["NLC"], errors="coerce").dropna().astype(int)
)
travel_nlcs = set(
    pd.to_numeric(df_travel["From_NLC"], errors="coerce").dropna().astype(int)
).union(
    set(pd.to_numeric(df_travel["To_NLC"], errors="coerce").dropna().astype(int))
)

# 3. Identify missing NLCs
missing_nlcs = all_station_nlcs - travel_nlcs
df_missing = df_stations[
    df_stations["NLC"].astype(int).isin(missing_nlcs)
].copy()
df_missing.to_csv("stations_missing_from_travel_times.csv", index=False)

# 4. Print summary metrics
print(f"Total stations in all_stations.csv: {len(df_stations):,}")
print(
    "Stations with valid travel time edges:"
    f" {len(all_station_nlcs) - len(missing_nlcs):,}"
)
print(f"Stations missing from travel times: {len(df_missing):,}\n")

# Display missing stations grouped by transport operator/joint network
if not df_missing.empty:
  print("Breakdown of missing stations by network joint type:")
  print(df_missing["JOINT"].value_counts())
  print("\nFirst 15 missing stations:")
  display(df_missing[["NLC", "STATION", "JOINT", "GATED"]])
else:
  print("All stations in all_stations.csv have travel time edges!")

Total stations in all_stations.csv: 860
Stations with valid travel time edges: 755
Stations missing from travel times: 105

Breakdown of missing stations by network joint type:
JOINT
NR       63
TfL      34
Joint     8
Name: count, dtype: int64

First 15 missing stations:


,NLC,STATION,JOINT,GATED
20,5667,Ashford Surrey,NR,Ungated
23,1090,Aylesbury Valley Parkway,NR,Gated
26,512,Balham LU,TfL,Gated
34,5089,Barnehurst,NR,Ungated
59,520,Bethnal Green LU,TfL,Gated
...,...,...,...,...
791,5128,Welling,NR,Gated
807,1421,West Hampstead LO,NR,Gated
808,758,West Hampstead LU,TfL,Gated
842,6844,Witham,NR,Gated
